# HelaGPT Fine-tuning with PEFT and LoRA

This notebook fine-tunes a language model using Parameter-Efficient Fine-Tuning (PEFT) with LoRA adapters.

In [ ]:
# Install required packages
!pip install transformers datasets peft accelerate bitsandbytes
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# Import libraries
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import json
import os

In [ ]:
# Load training configuration
with open('training_config.json', 'r') as f:
    config = json.load(f)

model_name = config['model_name']
dataset_path = config['dataset_path']
output_dir = config['output_dir']
training_config = config['training_config']

In [ ]:
# Load and prepare dataset
with open(dataset_path, 'r') as f:
    dataset = json.load(f)

# Convert to Hugging Face format
train_data = []
for example in dataset['examples']:
    train_data.append({
        'text': f"Human: {example['input']}\nAssistant: {example['output']}"
    })

hf_dataset = Dataset.from_list(train_data)
print(f"Loaded {len(hf_dataset)} training examples")

In [ ]:
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Add padding token if not present
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=training_config['lora_rank'],
    lora_alpha=training_config['lora_alpha'],
    lora_dropout=training_config['lora_dropout'],
    target_modules=["q_proj", "v_proj"]
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Tokenize dataset
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding=True,
        max_length=training_config['max_length']
    )

tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
# Training arguments
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=training_config['num_epochs'],
    per_device_train_batch_size=training_config['batch_size'],
    learning_rate=training_config['learning_rate'],
    logging_steps=10,
    save_steps=100,
    evaluation_strategy="no",
    save_total_limit=2,
    remove_unused_columns=False,
    push_to_hub=False
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer
)

In [ ]:
# Start training
print("Starting training...")
trainer.train()
print("Training completed!")

In [ ]:
# Save the fine-tuned model
trainer.save_model()
tokenizer.save_pretrained(output_dir)

# Test the model
def generate_response(prompt, max_length=100):
    inputs = tokenizer.encode(prompt, return_tensors="pt")
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_length=max_length,
            num_return_sequences=1,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Test with sample prompt
test_prompt = "Human: Help me organize my files\nAssistant:"
response = generate_response(test_prompt)
print(f"Test response: {response}")